<a href="https://colab.research.google.com/github/trivenipujar27/Machine-Learning/blob/main/messy_data_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
df=pd.read_csv("/content/messy_ecommerce_dataset_120rows.csv")
print(df)

     Order_ID      Customer_Name     Age                          Email  \
0    ORD-1045     Sam Winchester     155     sam.winchester@example.com   
1    ORD-1048       Charlie Kahn      28       charlie.kahn@example.com   
2    ORD-1005  George Winchester  32 yrs  george.winchester@example.com   
3    ORD-1056       Olivia Brown      45       olivia.brown@example.com   
4    ORD-1027     Hannah Wallace      22     hannah.wallace@example.com   
..        ...                ...     ...                            ...   
116  ORD-1107        Noah Neeson      28        noah.neeson@example.com   
117  ORD-1015      Charlie Smith      45      charlie.smith@example.com   
118  ORD-1093                NaN     NaN          tina.atreides@example   
119  ORD-1052      George Wright     NaN      george.wright@example.com   
120  ORD-1103       Tina Wallace     155       tina.wallace@example.com   

               Phone  Order_Date Product_Category Quantity Unit_Price  \
0                NaN  2024

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Order_ID           121 non-null    object
 1   Customer_Name      114 non-null    object
 2   Age                113 non-null    object
 3   Email              115 non-null    object
 4   Phone              96 non-null     object
 5   Order_Date         109 non-null    object
 6   Product_Category   107 non-null    object
 7   Quantity           114 non-null    object
 8   Unit_Price         115 non-null    object
 9   Discount           97 non-null     object
 10  Total_Sales        118 non-null    object
 11  City               112 non-null    object
 12  Membership_Status  97 non-null     object
 13  Payment_Method     107 non-null    object
dtypes: object(14)
memory usage: 13.4+ KB


In [ ]:
df.isna().sum()


,0
Order_ID,0
Customer_Name,7
Age,8
Email,6
Phone,25
Order_Date,12
Product_Category,14
Quantity,7
Unit_Price,6
Discount,24


In [ ]:
df['Order_ID']=df['Order_ID'].str.strip()
df['Order_ID']=np.where(
    df['Order_ID'].str.startswith('ORD-'),
    df['Order_ID'],
    'ORD-'+df['Order_ID']
)


In [ ]:
df.duplicated().sum()

np.int64(5)

In [ ]:
df=df.drop_duplicates()
df=df.drop_duplicates(subset=['Order_ID'],keep='first')

In [ ]:
df['Order_ID']

,Order_ID
0,ORD-1045
1,ORD-1048
2,ORD-1005
3,ORD-1056
4,ORD-1027
...,...
115,ORD-1072
116,ORD-1107
118,ORD-1093
119,ORD-1052


In [ ]:
df['Customer_Name']

,Customer_Name
0,Sam Winchester
1,Charlie Kahn
2,George Winchester
3,Olivia Brown
4,Hannah Wallace
...,...
115,Charlie Winchester
116,Noah Neeson
118,NaN
119,George Wright


In [ ]:
#customer_name
df['Customer_Name'] = df['Customer_Name'].replace(['N/A', 'Unknown'], np.nan)
df['Customer_Name'] = df['Customer_Name'].str.strip().str.title()
df['Customer_Name']

,Customer_Name
0,Sam Winchester
1,Charlie Kahn
2,George Winchester
3,Olivia Brown
4,Hannah Wallace
...,...
115,Charlie Winchester
116,Noah Neeson
118,NaN
119,George Wright


In [ ]:
df['Email']=df['Email'].str.strip().str.lower()
valid_email=df['Email'].str.contains('@',na=False) & df['Email'].str.contains('.com',regex=False,na=False)
df.loc[~valid_email,'Email']=np.nan

In [ ]:
df['Email']
df["Email"].isna().sum()

np.int64(16)

In [ ]:
digits=df['Phone'].str.replace(r'\D','',regex=True)
df['Phone']=np.where(digits.str.len()==11,digits.str[1:],digits)
digits=pd.Series(digits,index=df.index)

valid_phone=(digits.str.len()==10)&(digits!='0000000000')
df['Phone']=np.where(
    valid_phone,
    digits.str[:3]+'-'+digits.str[3:6]+'-'+digits.str[6:],
    np.nan
)

In [ ]:
df['Phone']

,Phone
0,NaN
1,920-302-8269
2,NaN
3,NaN
4,NaN
...,...
115,517-871-7421
116,261-270-8734
118,310-641-3194
119,574-743-9221


In [ ]:
df['Product_Category']=df['Product_Category'].str.strip().str.title().replace({
    'Home and Kitchen':'Home & Kitchen',
    'Beauty and Health':'Beauty & Health'
})
df['City']=df['City'].str.strip().str.upper().replace({
    'NY':'NEW YORK',
    'LA':'LOS ANGELES',
    'CHI':'CHICAGO'
}).str.title()

df['Membership_Status']=df['Membership_Status'].str.strip().str.title().replace({'Unknown':np.nan}).fillna('Unassigned')

df['Payment_Method']=df['Payment_Method'].str.strip().str.lower().replace({
    'cc':'credit card',
    'dc':'debit card'
}).fillna('Unassigned')

In [ ]:
df['Age'] = pd.to_numeric(df['Age'].astype(str).str.extract(r'(\d+)', expand=False), errors='coerce')

df.loc[df['Age'] <= 0, 'Age'] = np.nan
df.loc[df['Age'] > 120, 'Age'] = np.nan
print(df['Age'].isnull().sum())

median_age = df['Age'].median()
df['Age'] = df['Age'].fillna(median_age).astype(int)

0


In [ ]:
df['Quantity']=df['Quantity'].replace({
  'two':2,
  'three':3,
  'four':4,
  'five':5,
  'six':6,
  'seven':7,
  'eight':8,
  'nine':9,
  'ten':10,

})


In [ ]:
print(df['Quantity'].unique())
print(df['Quantity'].dtype)

[3 '0' '-1' '2' '10' '3' '4' '500' nan '5' '1' '-2' 2]
object


In [ ]:
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')
df.loc[df['Quantity'] <= 0, 'Quantity'] = np.nan
df.loc[df['Quantity'] == 500, 'Quantity'] = np.nan

In [ ]:
df['Unit_Price']=(
    df['Unit_Price'].str.replace('$','',regex=False).str.replace('USD','',regex=False)
)


In [ ]:
df['Unit_Price'] = df['Unit_Price'].str.replace(",", ".", regex=False)

In [ ]:
df['Unit_Price']=pd.to_numeric(df['Unit_Price'],errors='coerce')


In [ ]:
df.loc[df['Unit_Price']<0,"Unit_Price"]=np.nan

In [ ]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce', format='mixed')
valid_dates = (df['Order_Date'] >= '2020-01-01') & (df['Order_Date'] <= '2026-12-31')
df.loc[~valid_dates, 'Order_Date'] = pd.NaT
df['Order_Date'] = df['Order_Date'].dt.strftime('%Y-%m-%d')
df['Order_Date']

,Order_Date
0,2024-12-19
1,NaN
2,2024-05-26
3,NaN
4,2025-07-09
...,...
115,2025-09-01
116,2024-10-20
118,2025-01-09
119,2024-06-17


In [ ]:
disc_text = df["Discount"].str.lower()
had_percent = disc_text.str.contains("%", regex=False)
disc = pd.to_numeric(disc_text.str.replace("%", "", regex=False), errors="coerce")  # "None"/"" -> NaN

# "15%" -> 0.15 and plain 15 -> 0.15, but 0.10 stays 0.10
disc = disc.where(~(had_percent | (disc > 1)), disc / 100)

was_missing = disc.isna()
was_invalid = (disc < 0) | (disc > 1)          # e.g. -10% or 150%
disc = disc.where(~was_invalid)                # invalid -> NaN

df["Discount_Flag"] = "OK"
df.loc[was_missing, "Discount_Flag"] = "Missing_Imputed_0"
df.loc[was_invalid, "Discount_Flag"] = "Invalid_Set_0"
df["Discount"] = disc.fillna(0.0)              # missing/invalid -> 0.00
print(df["Discount_Flag"].value_counts())

Discount_Flag
OK                   76
Missing_Imputed_0    22
Invalid_Set_0        17
Name: count, dtype: int64


In [ ]:
df['Discount']


,Discount
0,0.20
1,0.10
2,0.00
3,0.20
4,0.00
...,...
115,0.10
116,0.00
118,0.20
119,0.10


In [ ]:
# Total_Sales = Quantity x Unit_Price x (1 - Discount)
df["Total_Sales"] = (df["Quantity"] * df["Unit_Price"] * (1 - df["Discount"])).round(2)
print(df[["Quantity", "Unit_Price", "Discount", "Total_Sales"]].head())


   Quantity  Unit_Price  Discount  Total_Sales
0       3.0   999999.99       0.2   2399999.98
1       NaN         NaN       0.1          NaN
2       NaN      376.04       0.0          NaN
3       2.0       43.62       0.2        69.79
4       2.0      418.12       0.0       836.24


In [ ]:
n = len(df)
df = df.dropna(subset=["Order_ID", "Quantity"])
print("Dropped for missing Order_ID/Quantity:", n - len(df)); n = len(df)

df = df[df["Quantity"] > 0]        # zero/negative quantity = returns or errors
print("Dropped for quantity <= 0            :", n - len(df)); n = len(df)

df = df[df["Quantity"] <= 100]     # 500 is an extreme bulk outlier -> removed
print("Dropped for bulk outliers (> 100)    :", n - len(df)); n = len(df)

df = df.dropna(subset=["Unit_Price"])   # Unit_Price is a critical field
print("Dropped for missing/invalid price    :", n - len(df))

df["Quantity"] = df["Quantity"].astype(int)


Dropped for missing Order_ID/Quantity: 45
Dropped for quantity <= 0            : 0
Dropped for bulk outliers (> 100)    : 0
Dropped for missing/invalid price    : 7


In [ ]:
# Total_Sales = Quantity x Unit_Price x (1 - Discount)
df["Total_Sales"] = (df["Quantity"] * df["Unit_Price"] * (1 - df["Discount"])).round(2)
print(df[["Quantity", "Unit_Price", "Discount", "Total_Sales"]].head())


   Quantity  Unit_Price  Discount  Total_Sales
0         3   999999.99       0.2   2399999.98
3         2       43.62       0.2        69.79
4         2      418.12       0.0       836.24
5        10       96.09       0.0       960.90
6         3      485.10       0.0      1455.30
